# A.4 Samvariation

Det här notebooken undersöker samvariationen mellan de ekonomiska variablerna och deras koppling till skadefrekvens:

- korrelationsmatris för `Omsattning`, `Forsakringsbelopp`, `Sjalvrisk` — både råskala och log-skala
- samband mellan storlek och skadefrekvens via decilanalys
- diskussion om vilka variabler som sannolikt fångar liknande underliggande riskstorlek

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

data_dir = Path("../../data")
df = pd.read_csv(data_dir / "Entreprenadförsäkring training.csv")

print(f"Träning: {df.shape[0]:,} rader")

## 1. Korrelation mellan ekonomiska variabler

Här studeras den linjära samvariationen mellan de ekonomiska variablerna, både i ursprunglig skala och efter log-transformering.

In [ ]:
economic_variables = df[["Omsattning", "Forsakringsbelopp", "Sjalvrisk"]]
pearson_corr = economic_variables.corr(method="pearson")

log_economic_variables = pd.DataFrame(
    {
        "log_Omsattning": np.log1p(df["Omsattning"]),
        "log_Forsakringsbelopp": np.log1p(df["Forsakringsbelopp"]),
        "log_Sjalvrisk": np.log1p(df["Sjalvrisk"]),
    }
)
log_pearson_corr = log_economic_variables.corr(method="pearson")

print("Pearson-korrelation (råskala):")
display(pearson_corr)

print("\nPearson-korrelation (log-skala):")
display(log_pearson_corr)

### Tolkning

Pearson-korrelation i träningsdatan:

- Omsättning vs Försäkringsbelopp: 0,435
- Omsättning vs Självrisk: 0,456
- Försäkringsbelopp vs Självrisk: 0,213

Med log-transformerade variabler blir sambanden ännu tydligare, särskilt mellan Omsättning och Försäkringsbelopp.

**Vilka variabler fångar liknande riskstorlek?** Omsättning och Försäkringsbelopp speglar båda företagets storlek — stora företag tenderar att ha både hög omsättning och höga försäkringsvärden. Självrisk är delvis kopplad till storlek men mäter även kundens riskbenägenhet. I en regressionsmodell kan detta ge:

- svårare tolkning av enskilda koefficienter
- instabilare skattningar
- behov av att jämföra modeller med olika variabeluppsättningar

Det är sannolikt att Omsättning och Försäkringsbelopp bär överlappande information om underliggande företagsstorlek.

## 2. Samband mellan storlek och skador

Här delas observationerna in i storleksgrupper (deciler) för att undersöka hur skadefrekvensen varierar med omsättning, försäkringsbelopp och självrisk.

In [ ]:
def decile_claim_rate(data: pd.DataFrame, col: str, label: str) -> pd.DataFrame:
    deciles = pd.qcut(data[col], q=10, duplicates="drop")
    return (
        data.assign(Decil=deciles)
        .groupby("Decil", observed=False)
        .agg(
            **{
                "Antal rader": (col, "size"),
                "Totala skador": ("AntalSkador", "sum"),
                "Exponeringsår": ("Duration", "sum"),
            }
        )
        .assign(**{"Skador per exponerat år": lambda x: x["Totala skador"] / x["Exponeringsår"]})
        .reset_index()
        .rename(columns={"Decil": f"{label}-decil"})
    )


turnover_summary = decile_claim_rate(df, "Omsattning", "Omsättning")
insured_summary = decile_claim_rate(df, "Forsakringsbelopp", "Försäkringsbelopp")
deductible_summary = decile_claim_rate(df, "Sjalvrisk", "Självrisk")

print("Omsättning:")
display(turnover_summary[[f"Omsättning-decil", "Skador per exponerat år"]])

print("\nFörsäkringsbelopp:")
display(insured_summary[[f"Försäkringsbelopp-decil", "Skador per exponerat år"]])

print("\nSjälvrisk:")
display(deductible_summary[[f"Självrisk-decil", "Skador per exponerat år"]])

### Tolkning

Tydligt mönster i decilanalysen:

- **Omsättning:** Högre omsättning hänger ihop med högre skadefrekvens. Klar monoton trend från lägsta till högsta decilen.
- **Försäkringsbelopp:** Samma riktning — högre belopp, högre frekvens.
- **Självrisk:** Svagare och mindre tydligt samband.

Större företag och större försäkrade värden verkar ha högre risk. Det är affärsmässigt rimligt — större projekt innebär mer komplexitet och fler tillfällen till skador. Modelleringen måste dock hantera att Omsättning och Försäkringsbelopp bär liknande information (se korrelationsanalysen ovan).